In [35]:
import pickle
import pandas as pd

columns = ["question", "entities", "mask", "answer_masked", "answer_unmasked"]

df = pd.read_csv('baf_complete.tsv', sep="\t")
df.columns = columns

df_no = pd.read_csv('baf_no_synonyms.tsv', sep="\t")
df_no.columns = columns

df_gpt = pd.read_csv('gpt.tsv', sep="\t")
df_gpt.columns = columns


In [3]:
#Download dataset from: https://drive.google.com/drive/folders/0B-36Uca2AvwhTWVFSUZqRXVtbUE?resourcekey=0-kdv6ho5KcpEXdI2aUdLn_g
dataset_folder = ""
dataset_file = dataset_folder + "METAQA/kb.txt"

In [4]:
def load_qa_data(file):
    questions = []
    answers = []
    
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()  # remove newline/whitespace
            if not line:  # skip empty lines
                continue
            
            # split into question and answer part
            q, a = line.split("\t", 1)
            
            # store question
            questions.append(q)
            
            # split answers by "|" and strip whitespace
            answers.append([ans.strip() for ans in a.split("|")])
    return questions, answers 

In [5]:
#Download dataset from: https://drive.google.com/drive/folders/0B-36Uca2AvwhTWVFSUZqRXVtbUE?resourcekey=0-kdv6ho5KcpEXdI2aUdLn_g
test_file1 = dataset_folder+"METAQA/1-hop/qa_test.txt"
test_file2 = dataset_folder+"METAQA/2-hop/qa_test.txt"
test_file3 = dataset_folder+"METAQA/3-hop/qa_test.txt"

test_questions1, test_answers1 = load_qa_data(test_file1)
test_questions2, test_answers2 = load_qa_data(test_file2)
test_questions3, test_answers3 = load_qa_data(test_file3)

In [6]:
print('Number of test questions:', len(test_questions1) + len(test_questions2) + len(test_questions3))

Number of test questions: 39093


In [7]:
import re

def escape_quotes_in_brackets(text):
    def replacer(match):
        inside = match.group(1)
        # Escape all single quotes inside the brackets
        return "[" + inside.replace("'", r"\'") + "]"
    
    # Find [ ... ] groups and apply replacer
    return re.sub(r"\[([^\]]*)\]", replacer, text)

def questions_stats(test_question, test_answer):
    questions = {}
    unique_questions = []
    unique_answers = []
    for question, answer in zip(test_question, test_answer):
        unique_question =  re.sub(r"\[.*?\]", "", question)
    
        if unique_question in questions:
            questions[unique_question] += 1
        else:
            questions[unique_question] = 1
            # Add spaces around only , . ! ?
            text_spaced = re.sub(r',(?![^\[]*\])', r' , ', question)
            # Remove extra spaces
            text_spaced = re.sub(r'\s+', ' ', text_spaced).strip()
            unique_questions.append(text_spaced)
            unique_answers.append(answer)
    
    less_frequent = len(test_question)
    more_frequent = 0
    for q in questions:
        if questions[q] < less_frequent:
            less_frequent = questions[q]
        if questions[q] > more_frequent:
            more_frequent = questions[q]

    return ({
        'len_questions' : len(test_question),
        'unique_questions' : len(questions),
        'less_frequent' : less_frequent,
        'more_frequent' : more_frequent
    }, unique_questions, unique_answers)    

In [8]:
stats1, unique_questions1, unique_answers1 = questions_stats(test_questions1, test_answers1)
stats2, unique_questions2, unique_answers2 = questions_stats(test_questions2, test_answers2)
stats3, unique_questions3, unique_answers3 = questions_stats(test_questions3, test_answers3)

sample_questions = unique_questions1 + unique_questions2 + unique_questions3
sample_answers = unique_answers1 + unique_answers2 + unique_answers3

print("Number of unique questions:", len(sample_questions))

Number of unique questions: 503


In [11]:
from neo4j import GraphDatabase

# Server details
URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "neo4j123")
DATABASE_NAME = "neo4j"

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Connection with the server stablished successfully.")


Connection with the server stablished successfully.


In [17]:
from tqdm import tqdm
def run_cypher(baf_answers, URI, AUTH):
    #Run CYPHER queries
    driver = GraphDatabase.driver(URI, auth=AUTH)
    
    results = []
    for answer in tqdm(baf_answers):
        try:
            records, summary, keys = driver.execute_query(answer,database_=DATABASE_NAME)
            results.append((records, summary, keys))
        except:
            results.append(([], None, None))
    driver.close()

    parsed_results = []
    for result in results:
        data = result[0]  # this is a list of dictionaries
        parsed_results.append([v for d in data for v in d.values()])
    
    return parsed_results

In [20]:
from typing import List

def accuracy_score(ground_truth: List[List[str]], predictions: List[List[str]]):
    wrong = []
    correct = 0
    total = len(ground_truth)
    
    for i, (gt, pred) in enumerate(zip(ground_truth, predictions)):
        # Flatten pred if it contains lists
        flattened = [x for item in pred for x in (item if isinstance(item, list) else [item])]
        
        if set(gt) == set(flattened):
            correct += 1
        else:
            wrong.append([i, gt, pred])

    acc = correct / total if total > 0 else 0.0
    return acc, wrong

### AskSafely

In [ ]:
parsed_results = run_cypher(list(df['answer_unmasked']), URI, AUTH)

In [54]:
acc, wrong_answers = accuracy_score(sample_answers[1:], parsed_results)
print("Accuracy:", acc_no)
print("#Errors:", len(wrong_answers_no))

Accuracy: 0.7370517928286853
#Errors: 132


In [55]:
print("True accuracy: ", ((502 - len(wrong_answers) + 31) / 502))

True accuracy:  0.8685258964143426


In [33]:
for wrong in wrong_answers:
    id_wrong = wrong[0]
    print(id_wrong, df['question'][id_wrong])
    print(df['answer_unmasked'][id_wrong])
    print(wrong[1])
    print(wrong[2])
    print()
        

20 what did [Garth Jennings] direct
MATCH (m:Movie)-[r]-(d:Director) WHERE toLower(m.name) = toLower('Garth Jennings') RETURN DISTINCT d.name
["The Hitchhiker's Guide to the Galaxy", 'Son of Rambow']
[]

29 the film [Pennies from Heaven] starred which actors
MATCH (m:Movie)-[r]-(a:Actor) WHERE toLower(m.name) = toLower('Pennies from Heaven') RETURN DISTINCT a.name
['Bing Crosby', 'Louis Armstrong', 'Madge Evans', 'Edith Fellows']
['Madge Evans', 'Bing Crosby', 'Steve Martin', 'Louis Armstrong', 'Bernadette Peters', 'Edith Fellows', 'Jessica Harper']

63 what was the genre of the film [Bad Boys]
MATCH (m:Movie)-[r]-(g:Genre) WHERE toLower(m.name) = toLower('Bad Boys') RETURN DISTINCT g.name
['Drama', 'Crime']
['Drama', 'Comedy', 'Crime', 'Action']

80 describe [The Butcher Boy]
MATCH (m:Movie) WHERE any(tag IN m.has_tags WHERE toLower(tag) = toLower('The Butcher Boy')) RETURN DISTINCT m.name
['neil jordan']
[]

84 can you give a few words describing [The Iron Horse]
MATCH (m:Movie) WHER

### No Synonym

In [39]:
parsed_results_no = run_cypher(list(df_no['answer_unmasked']), URI, AUTH)

100%|█████████████████████████████████████████████████████████████████████████████████| 502/502 [00:27<00:00, 18.12it/s]


In [45]:
acc_no, wrong_answers_no = accuracy_score(sample_answers[1:], parsed_results_no)
print("Accuracy:", acc_no)
print("#Errors:", len(wrong_answers_no))

Accuracy: 0.7370517928286853
#Errors: 132


In [51]:
print("True accuracy: ", ((502 - len(wrong_answers_no) + 33) / 502))

True accuracy:  0.8027888446215139


In [41]:
for wrong in wrong_answers_no:
    id_wrong = wrong[0]
    print(id_wrong, df_no['question'][id_wrong])
    print(df_no['answer_unmasked'][id_wrong])
    print(wrong[1])
    print(wrong[2])
    print()
        

29 the film [Pennies from Heaven] starred which actors
MATCH (m:Movie)-[r]-(a:Actor) WHERE toLower(m.name) = toLower('Pennies from Heaven') RETURN DISTINCT a.name
['Bing Crosby', 'Louis Armstrong', 'Madge Evans', 'Edith Fellows']
['Madge Evans', 'Bing Crosby', 'Steve Martin', 'Louis Armstrong', 'Bernadette Peters', 'Edith Fellows', 'Jessica Harper']

63 what was the genre of the film [Bad Boys]
MATCH (m:Movie)-[r]-(g:Genre) WHERE toLower(m.name) = toLower('Bad Boys') RETURN DISTINCT g.name
['Drama', 'Crime']
['Drama', 'Comedy', 'Crime', 'Action']

73 which topics is movie [Burden of Dreams] about
MATCH (m:Movie) WHERE toLower(m.name) = toLower('Burden of Dreams') UNWIND coalesce(m.has_tags, []) AS topic RETURN DISTINCT topic UNION MATCH (m:Movie) WHERE toLower(m.name) = toLower('Burden of Dreams') UNWIND coalesce(m.has_genre, []) AS topic RETURN DISTINCT topic UNION MATCH (m:Movie)-[r]-(g:Genre) WHERE toLower(m.name) = toLower('Burden of Dreams') RETURN DISTINCT g.name AS topic
['bd-r'

### GPT

In [42]:
parsed_results_gpt = run_cypher(list(df_gpt['answer_unmasked']), URI, AUTH)

 64%|███████████████████████████████████████████████████▍                             | 319/502 [00:08<00:04, 40.59it/s]Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=1, column=106, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 1, 'column': 106}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (w1:Writer)-[r1]-(m:Movie)-[r2]-(w2:Writer) WHERE toLower(w1.name) = toLower('Elizabeth Kata') AND id(w2) <> id(w1) RETURN DISTINCT w2.name"
Received notification from DBMS server: <Gq

In [52]:
acc_gpt, wrong_answers_gpt = accuracy_score(sample_answers[1:], parsed_results_gpt)
print("Accuracy:", acc_gpt)
print("#Errors:", len(wrong_answers_gpt))

Accuracy: 0.8386454183266933
#Errors: 81


In [44]:
for wrong in wrong_answers_gpt:
    id_wrong = wrong[0]
    print(id_wrong, df_gpt['question'][id_wrong])
    print(df_gpt['answer_unmasked'][id_wrong])
    print(wrong[1])
    print(wrong[2])
    print()
        

14 can you name a film directed by [Paul Wendkos]
MATCH (m:Movie)-[r]-(d:Director) WHERE toLower(d.name) = toLower('Paul Wendkos') RETURN DISTINCT m.name LIMIT 1
['Gidget', 'Guns of the Magnificent Seven', 'Attack on the Iron Coast', 'Face of a Fugitive']
['Guns of the Magnificent Seven']

24 who acted in the movie [Singin' in the Rain]
MATCH (m:Movie)-[r]-(a:Actor) WHERE toLower(m.name) = toLower('Singin'' in the Rain') RETURN DISTINCT a.name
['Gene Kelly', 'Debbie Reynolds', 'Jean Hagen', "Donald O'Connor"]
[]

29 the film [Pennies from Heaven] starred which actors
MATCH (m:Movie)-[r]-(a:Actor) WHERE toLower(m.name) = toLower('Pennies from Heaven') RETURN DISTINCT a.name
['Bing Crosby', 'Louis Armstrong', 'Madge Evans', 'Edith Fellows']
['Madge Evans', 'Bing Crosby', 'Steve Martin', 'Louis Armstrong', 'Bernadette Peters', 'Edith Fellows', 'Jessica Harper']

44 what sort of film is [The Well-Digger's Daughter]
MATCH (m:Movie)-[r]-(g:Genre) WHERE toLower(m.name) = toLower('The Well-Dig

In [53]:
print("True accuracy: ", ((502 - len(wrong_answers_gpt) + 51) / 502))

True accuracy:  0.9402390438247012
